In [1]:
import pandas as pd

df = pd.read_csv("merged_deduplicated.csv", encoding="utf-8-sig")

df["record_id"] = range(1, len(df) + 1)

for col in ["Title", "Abstract", "Keywords"]:
    df[col] = df[col].fillna("").astype(str)

df["title_text"] = df["Title"].str.lower()
df["screening_text"] = (
    df["Title"] + " " + df["Abstract"]
).str.lower()

df["screening_decision"] = pd.NA
df["exclusion_reason"] = pd.NA

In [2]:

import re
import pandas as pd

# Example terms and weights: replace after pilot screening

positive_terms = {
    # Performance domains
    "singing voice": 10,
    "singing voice analysis": 10,
    "singing voice detection": 10,
    "solo singing": 10,
    "vocal performance": 9,
    "singing": 8,
    "vocal": 8,
    "singer": 8,
    "choir": 10,
    "choral": 10,
    "choral singing": 10,
    "ensemble singing": 10,
    "vocal ensemble": 10,
    "polyphonic singing": 10,
    "multi-singer": 10,
    "multiple singers": 9,
    "instrumental performance": 10,
    "instrumental music": 9,
    "instrument recognition": 10,
    "musical instrument": 8,
    "piano transcription": 10,
    "guitar transcription": 10,
    "orchestral music": 10,
    "polyphonic music": 10,
    "music performance": 8,
    "expressive performance": 5,
    "ensemble": 2,

    # MIR tasks
    "music information retrieval": 10,
    "music retrieval": 9,
    "automatic music transcription": 10,
    "music transcription": 10,
    "audio transcription": 8,
    "polyphonic transcription": 10,
    "multipitch estimation": 10,
    "multi-pitch estimation": 10,
    "pitch estimation": 9,
    "pitch detection": 9,
    "fundamental frequency estimation": 8,
    "melody extraction": 10,
    "predominant melody": 9,
    "singing voice separation": 10,
    "music source separation": 10,
    "source separation": 8,
    "vocal separation": 10,
    "instrument separation": 10,
    "audio-to-score alignment": 10,
    "score following": 9,
    "music alignment": 9,
    "onset detection": 9,
    "note onset detection": 10,
    "beat tracking": 7,
    "tempo estimation": 7,
    "rhythm analysis": 7,
    "chord recognition": 8,
    "music structure analysis": 7,
    "instrument classification": 9,
    "music classification": 6,
    "music tagging": 5,
    "audio tagging": 4,
    "music similarity": 5,
    "music emotion recognition": 4,
    "performance assessment": 9,
    "singing assessment": 10,
    "intonation assessment": 9,
    "timbre analysis": 7,

    # Audio representations
    "audio signal processing": 8,
    "music signal processing": 9,
    "musical audio": 8,
    "audio representation": 6,
    "spectrogram": 5,
    "mel-spectrogram": 6,
    "short-time fourier transform": 6,
    "constant-q transform": 7,
    "chromagram": 6,
    "chroma features": 5,
    "harmonic-percussive separation": 8,
    "harmonic analysis": 5,
    "time-frequency representation": 6,
    "audio embedding": 6,
    "acoustic features": 5,
    "waveform": 4,
    "spectral features": 5,

    # Models and architectures
    "machine learning": 3,
    "deep learning": 4,
    "neural network": 3,
    "convolutional neural network": 4,
    "recurrent neural network": 4,
    "long short-term memory": 4,
    "transformer": 4,
    "attention mechanism": 3,
    "self-supervised learning": 5,
    "contrastive learning": 4,
    "transfer learning": 5,
    "domain adaptation": 6,
    "foundation model": 4,
    "pretrained model": 3,
    "pre-trained model": 3,
    "representation learning": 4,
    "encoder-decoder": 4,
    "u-net": 4,
    "diffusion model": 3,
    "hybrid model": 3,
    "signal processing": 3,

    # RQ2 challenges
    "generalization": 5,
    "generalisation": 5,
    "cross-dataset": 6,
    "cross-domain": 6,
    "domain shift": 6,
    "robustness": 5,
    "data scarcity": 6,
    "limited data": 4,
    "dataset": 3,
    "benchmark": 5,
    "reproducibility": 6,
    "replication": 4,
    "out-of-distribution": 5,
    "unseen instruments": 8,
    "unseen singers": 8,
    "real-world recordings": 6,
    "polyphonic mixtures": 7,
    "overlapping sources": 7,
    "annotation quality": 5,
    "ground truth": 4,

    # Broad signals
    "mir": 2,
    "music analysis": 5,
    "computational music": 5,
    "music audio": 7,
    "music understanding": 6,
    "music dataset": 6,
    "music benchmark": 7,
}

negative_terms = {
    # Speech and nonmusical audio
    "automatic speech recognition": 9,
    "speech recognition": 8,
    "speaker verification": 9,
    "speaker identification": 9,
    "speaker recognition": 9,
    "text-to-speech": 8,
    "speech synthesis": 7,
    "dialogue system": 9,
    "spoken language understanding": 9,
    "speech translation": 9,
    "voice conversion": 4,
    "speech enhancement": 3,
    "speech separation": 3,
    "speech processing": 3,
    "environmental sound classification": 7,
    "acoustic scene classification": 6,
    "sound event detection": 4,

    # Biomedical
    "parkinson's disease": 8,
    "alzheimer's disease": 8,
    "clinical diagnosis": 7,
    "medical diagnosis": 7,
    "healthcare": 6,
    "patient monitoring": 7,
    "disease detection": 7,
    "biomarker": 6,
    "dysphonia": 5,
    "pathology": 3,
    "hospital": 5,
    "hearing aid": 7,
    "cochlear implant": 7,
    "audiology": 7,
    "hearing impairment": 6,
    "hearing loss": 6,
    "medical": 2,
    "clinical": 2,

    # Recommendation and consumer behavior
    "recommender system": 8,
    "music recommendation": 8,
    "playlist generation": 9,
    "playlist continuation": 9,
    "collaborative filtering": 8,
    "user preference": 6,
    "listening preference": 7,
    "consumer behavior": 9,
    "music consumption": 8,
    "personalized recommendation": 8,
    "personalized playlist": 9,
    "music discovery": 4,

    # Business and industry
    "music marketing": 10,
    "music licensing": 9,
    "streaming revenue": 10,
    "streaming platform": 7,
    "music popularity": 7,
    "chart prediction": 9,
    "music industry": 7,
    "music economics": 10,
    "copyright management": 8,
    "royalty distribution": 10,
    "social media engagement": 9,
    "consumer engagement": 9,

    # Psychology and education
    "music cognition": 3,
    "emotion perception": 4,
    "listener study": 3,
    "perceptual study": 2,
    "survey study": 6,
    "questionnaire": 5,
    "music education": 2,
    "music pedagogy": 3,
    "psychological well-being": 8,
    "music preference survey": 8,
    "behavioral experiment": 3,

    # Musicology
    "historical music analysis": 7,
    "music history": 8,
    "historical musicology": 8,
    "ethnomusicology": 4,
    "music evolution": 4,
    "style evolution": 4,
    "music theory education": 6,
    "music criticism": 9,
    "cultural analysis": 6,
    "musicology": 2,

    # Symbolic-only and non-audio
    "optical music recognition": 6,
    "sheet music recognition": 7,
    "score image recognition": 7,
    "symbolic music generation": 8,
    "midi-only": 8,
    "symbolic-only": 9,
    "symbolic music modeling": 5,
    "score engraving": 9,
    "notation recognition": 6,
    "music notation software": 8,
    "lyrics classification": 7,
    "lyric sentiment analysis": 8,

    # Generation and metadata
    "music generation": 3,
    "music synthesis": 3,
    "singing voice synthesis": 3,
    "text-to-music generation": 7,
    "symbolic composition": 7,
    "algorithmic composition": 6,
    "metadata tagging": 5,
    "artist tagging": 6,
    "semantic tagging": 3,
    "genre tagging": 2,
    "mood detection": 3,
    "music therapy": 7,
}


def count_matches(text, term):
    pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"
    return len(re.findall(pattern, text, flags=re.I))

def weighted_total(text, terms, word_count):
    total = 0.0

    for term, weight in terms.items():
        matches = count_matches(text, term)
        term_words = len(term.split())

        normalized_matches = (
            matches * term_words / word_count
        )

        total += (weight / 10) * normalized_matches

    return total

def calculate_term_score(text):
    word_count = max(len(text.split()), 1)

    positive = weighted_total(
        text, positive_terms, word_count
    )
    negative = weighted_total(
        text, negative_terms, word_count
    )

    return positive - (1.2 * negative)

df["term_score"] = df["screening_text"].apply(
    calculate_term_score
)

df = df.sort_values(
    "term_score", ascending=False
)

df.to_csv("mir_ranked_records.csv", index=False)


In [3]:
# ---------------------------------------------------------------------------
# Split mir_ranked_records.csv into three batches of 300 records each
# ---------------------------------------------------------------------------

import pandas as pd

ranked = pd.read_csv("mir_ranked_records.csv", encoding="utf-8-sig")

batch_size = 300
total      = len(ranked)

batch1 = ranked.iloc[0           : batch_size      ].copy()
batch2 = ranked.iloc[batch_size  : batch_size * 2  ].copy()
batch3 = ranked.iloc[batch_size * 2 : batch_size * 3].copy()

batch1.to_csv("mir_ranked_batch1_1-300.csv",   index=False, encoding="utf-8-sig")
batch2.to_csv("mir_ranked_batch2_301-600.csv", index=False, encoding="utf-8-sig")
batch3.to_csv("mir_ranked_batch3_601-900.csv", index=False, encoding="utf-8-sig")

print(f"Total ranked records : {total}")
print(f"Batch 1  (records   1-300) : {len(batch1)} records  ->  mir_ranked_batch1_1-300.csv")
print(f"Batch 2  (records 301-600) : {len(batch2)} records  ->  mir_ranked_batch2_301-600.csv")
print(f"Batch 3  (records 601-900) : {len(batch3)} records  ->  mir_ranked_batch3_601-900.csv")


Total ranked records : 9584
Batch 1  (records   1-300) : 300 records  ->  mir_ranked_batch1_1-300.csv
Batch 2  (records 301-600) : 300 records  ->  mir_ranked_batch2_301-600.csv
Batch 3  (records 601-900) : 300 records  ->  mir_ranked_batch3_601-900.csv


In [ ]:
# =============================================================================
# PHASE 1 — Step 4: Consolidate, Filter & Merge Screening Batches
# =============================================================================
import os
import pandas as pd

NB_DIR = os.path.dirname(os.path.abspath('phase1.ipynb'))

# ---------------------------------------------------------------------------
# 1. Locate and load screening result CSVs
# ---------------------------------------------------------------------------
BATCH_FILES = {
    'Batch 1': os.path.join(NB_DIR, 'batch 1 - Sheet1.csv'),
    'Batch 2': os.path.join(NB_DIR, 'batch 2 - mir_screening_results_v2.csv.csv'),
    'Batch 3': os.path.join(NB_DIR, 'batch 3 - Sheet1.csv'),
}

# Canonical column names we need from every batch
REQUIRED_COLS = ['paper_id', 'title', 'classification', 'screening_decision', 'reasoning']

# Column aliases: maps any variant name -> canonical name
COL_ALIASES = {
    'record_id':           'paper_id',
    'id':                  'paper_id',
    'Title':               'title',
    'TITLE':               'title',
    'Classification':      'classification',
    'CLASSIFICATION':      'classification',
    'Screening_Decision':  'screening_decision',
    'ScreeningDecision':   'screening_decision',
    'Decision':            'screening_decision',
    'SCREENING_DECISION':  'screening_decision',
    'Reasoning':           'reasoning',
    'REASONING':           'reasoning',
    'reason':              'reasoning',
}

batch_frames = []
for batch_name, fpath in BATCH_FILES.items():
    if not os.path.exists(fpath):
        print(f'  [WARNING] {batch_name} not found: {fpath}')
        continue
    df_b = pd.read_csv(fpath, encoding='utf-8-sig')
    # Normalise column names
    df_b = df_b.rename(columns=COL_ALIASES)
    df_b.columns = [c.strip().lower().replace(' ', '_') for c in df_b.columns]
    df_b['_source_batch'] = batch_name
    batch_frames.append(df_b)
    print(f'  Loaded {batch_name}: {len(df_b):,} rows | cols: {list(df_b.columns)}')

assert batch_frames, 'No batch files found — check file paths above.'

master_raw = pd.concat(batch_frames, ignore_index=True)
print(f'\nMaster (all batches combined): {len(master_raw):,} rows')

# ---------------------------------------------------------------------------
# 2. Filter — keep only RETAIN decisions, drop EXCLUDE
# ---------------------------------------------------------------------------
exclude_mask = master_raw['screening_decision'].str.upper().str.strip().isin(
    ['EXCLUDE', 'EXCLUDED', 'EXCL']
)
n_excluded = exclude_mask.sum()
retained = master_raw[~exclude_mask].copy()
print(f'  Excluded : {n_excluded:,}')
print(f'  Retained : {len(retained):,}')

# ---------------------------------------------------------------------------
# 3. Merge retained papers with mir_ranked_records (full bibliographic data)
# ---------------------------------------------------------------------------
ranked_path = os.path.join(NB_DIR, 'mir_ranked_records.csv')
ranked = pd.read_csv(ranked_path, encoding='utf-8-sig')

# Normalise ranked join key to 'paper_id'
if 'record_id' in ranked.columns and 'paper_id' not in ranked.columns:
    ranked = ranked.rename(columns={'record_id': 'paper_id'})

# Drop columns from ranked that will be overridden by screening results
drop_from_ranked = ['screening_decision', 'exclusion_reason']
ranked_clean = ranked.drop(columns=[c for c in drop_from_ranked if c in ranked.columns])

# Merge on paper_id
merged = retained.merge(
    ranked_clean,
    on='paper_id',
    how='left',
    suffixes=('', '_ranked')
)

# Add enrichment columns
merged['screening_classification'] = merged['classification'].str.upper().str.strip()
merged['screening_decision']       = 'RETAIN'

# Reorder: put core bibliographic columns first
bib_cols   = ['Title', 'Authors', 'Year', 'DOI', 'Abstract', 'Keywords',
               'Publication Venue', 'Document Type', 'Source Database', 'PDF Link']
meta_cols  = ['paper_id', 'term_score', 'screening_classification',
               'screening_decision', 'classification', 'reasoning', '_source_batch']
other_cols = [c for c in merged.columns
              if c not in bib_cols + meta_cols]

ordered = [c for c in bib_cols + meta_cols + other_cols if c in merged.columns]
final_df = merged[ordered]

# ---------------------------------------------------------------------------
# 4. Export
# ---------------------------------------------------------------------------
out_path = os.path.join(NB_DIR, 'mir_included_papers_master.csv')
final_df.to_csv(out_path, index=False, encoding='utf-8-sig')

# ---------------------------------------------------------------------------
# Summary verification
# ---------------------------------------------------------------------------
print('\n' + '='*65)
print('  PIPELINE SUMMARY')
print('='*65)
print(f'  Total records processed (all batches) : {len(master_raw):,}')
for batch_name in BATCH_FILES:
    n = len([f for f in batch_frames if f['_source_batch'].iloc[0] == batch_name][0]) if batch_frames else 0
    sub = master_raw[master_raw['_source_batch'] == batch_name]
    print(f'    {batch_name}: {len(sub):,} rows')
print(f'  Total EXCLUDED                        : {n_excluded:,}')
print(f'  Total RETAINED (before merge)         : {len(retained):,}')
print(f'  Total in mir_included_papers_master   : {len(final_df):,}')
unmatched = final_df['Title'].isna().sum()
if unmatched:
    print(f'  [WARNING] {unmatched} retained records had no match in ranked master.')
print(f'\n  Output: {out_path}')
print('='*65)
print('\nFinal columns:')
print(list(final_df.columns))
print('\nFirst 5 rows preview:')
display(final_df[['Title','Authors','Year','DOI','screening_classification',
                   'screening_decision','term_score']].head(5))
